# Knowledge Graph Builder

**Scenario:** You're building an internal knowledge base for a company wiki. The content lives in unstructured articles — employee bios, project descriptions, partnership announcements. You need to turn this into a structured, queryable knowledge graph that shows who works on what, which teams own which products, and how projects relate.

But the real payoff isn't just the graph itself — it's what you do with it. In sections 7 and 8, we'll show how Relation Extraction results make your vector search dramatically smarter: filtering by entity metadata and embedding structural knowledge directly into vectors.

**What we'll build:**
1. Extract entities from a collection of wiki-style articles
2. Feed entities + text into the Relation Extraction service to discover relationships
3. Resolve duplicate entities (e.g., "John Smith" and "J. Smith" → same person)
4. Generate knowledge graph in custom, property_graph, and RDF formats
5. Run the full pipeline (Entity Extraction → Relation Extraction) as a single Pipeline job
6. Visualize the resulting graph
7. Use Relation Extraction output as metadata filters for vector search (Pinecone/Qdrant/Weaviate pattern)
8. Graph-augmented embeddings: embed structural knowledge for smarter retrieval

**Services used:** Entity Extraction, Relation Extraction, Embedding, Pipelines

**Prerequisites:**
- `pip install latence numpy`
- `LATENCE_API_KEY` environment variable set
- Optional: `pip install networkx matplotlib` (for graph visualization)

In [ ]:
import sys
sys.path.insert(0, "..")
from _test_utils import setup_notebook, print_section, print_subsection, print_success, time_operation

import numpy as np

client = setup_notebook()

## Sample Data: Company Wiki Articles

A set of realistic internal wiki articles that reference overlapping people, teams, and projects.

In [ ]:
articles = [
    {
        "title": "Project Aurora - Q1 Launch",
        "text": (
            "Project Aurora is our next-generation recommendation engine, led by Dr. Maria Santos "
            "from the AI Research team. The project was greenlit by CTO James Wright in November 2024 "
            "with a budget of $2.4M. Aurora uses transformer-based models trained on our proprietary "
            "dataset. The engineering team — led by Senior Engineer Raj Patel — built the inference "
            "pipeline on Kubernetes. Beta testing begins in March 2025 with partner company DataFlow Inc."
        ),
    },
    {
        "title": "Team Spotlight: AI Research",
        "text": (
            "The AI Research team, headed by Dr. Maria Santos, focuses on applied machine learning "
            "for product features. Key members include Li Wei (NLP specialist), Priya Sharma "
            "(computer vision), and Alex Kim (MLOps). The team reports to CTO James Wright and "
            "collaborates closely with the Platform Engineering group led by Raj Patel. Recent "
            "achievements include Project Aurora and the internal search ranking system codenamed Compass."
        ),
    },
    {
        "title": "Partnership Announcement: DataFlow Inc.",
        "text": (
            "We're excited to announce a strategic partnership with DataFlow Inc., a San Francisco-based "
            "data analytics company founded by CEO Rebecca Torres. DataFlow will integrate our "
            "recommendation engine (Project Aurora) into their enterprise analytics platform. The deal, "
            "valued at $1.8M over 18 months, was negotiated by VP of Business Development Sarah Lin "
            "and DataFlow's CTO Marcus Johnson. Implementation kicks off in Q2 2025."
        ),
    },
]

print(f"Loaded {len(articles)} articles")
for a in articles:
    print(f"  - {a['title']} ({len(a['text'])} chars)")

## 1. Extract Entities from Each Article

First pass: identify all the people, organizations, projects, dates, and amounts mentioned across articles.

In [ ]:
# TODO: Extract entities from each article using label_mode="generated"
# Collect all entities across articles for the next step

## 2. Build Relationships (Relation Extraction)

Now feed the entities + text into the Relation Extraction service. It discovers how entities relate: who leads what, who works with whom, what's connected to what.

In [ ]:
# TODO: Build knowledge graph for each article, then combine
# Show relations like: Dr. Maria Santos --[leads]--> Project Aurora

## 3. Entity Resolution

"Dr. Maria Santos" in article 1 and "Dr. Maria Santos" in article 2 are the same person. Entity resolution merges duplicates across the combined graph.

In [ ]:
# TODO: Re-run ontology with resolve_entities=True on combined text
# Compare entity counts before/after resolution

## 4. Output Formats

Generate the graph in different formats depending on your downstream use:
- `custom` — Python-friendly dict with nodes/edges
- `property_graph` — for graph databases (Neo4j, etc.)
- `rdf` — for semantic web / linked data systems

In [ ]:
# TODO: Generate graph in all 3 formats, show structure of each

## 5. Pipeline Version (Single API Call)

Instead of calling Entity Extraction then Relation Extraction separately, use a Pipeline to chain them in one async job.

In [ ]:
# TODO: Build and execute a PipelineBuilder().extraction().ontology() pipeline
# Compare output with the manual two-step approach

## 6. Visualize the Graph

Turn the knowledge graph into a visual network diagram.

In [ ]:
# TODO: Use networkx + matplotlib to visualize (if installed)
# Otherwise, print a text-based graph representation

---

# Part 2: Why This Matters — Knowledge Graphs for Smarter Vector Search

Building a knowledge graph is interesting, but the real value is what you *do* with it. The two sections below show how Relation Extraction output directly improves the quality of vector-based retrieval — the kind that powers every RAG pipeline, enterprise search, and AI assistant.

Plain HNSW vector search treats every chunk as a bag of semantics. It has no concept of *who* is mentioned, *what* they're related to, or *which* entities appear. Relation Extraction gives you that structure, and there are two practical ways to use it.

## 7. Metadata-Enriched Vectors

**The problem:** You search for "Who leads Project Aurora?" and your vector index returns all three articles — because they all mention Aurora, leadership, or people. The semantic similarity scores are close. There's no way to narrow the search structurally.

**The solution:** Attach Relation Extraction-derived metadata to each vector when you upsert to your index. At query time, use metadata filters to scope the search *before* cosine similarity runs. This is exactly how Pinecone, Qdrant, Weaviate, and Milvus metadata filtering works — the Relation Extraction output maps directly to filter dimensions.

In [ ]:
# Step 1: Build metadata from ontology results for each article
#
# For each article, we already have extraction + ontology results from sections 1-3.
# Extract structured metadata:

def build_vector_metadata(title, entities, relations):
    """Convert ontology output into vector DB metadata."""
    entity_types = list({e.label for e in entities})
    entity_names = list({e.text for e in entities})
    relation_strings = [
        f"{r.source_entity} --{r.relation_type}--> {r.target_entity}"
        for r in relations
    ]
    return {
        "title": title,
        "entity_types": entity_types,
        "entity_names": entity_names,
        "relations": relation_strings,
        "has_people": "PERSON" in entity_types,
        "has_orgs": "ORGANIZATION" in entity_types or "ORG" in entity_types,
        "has_monetary": any(t in entity_types for t in ["MONETARY", "MONEY", "AMOUNT"]),
        "person_names": [e.text for e in entities if e.label == "PERSON"],
    }

# TODO: Apply to each article's ontology results
# metadata_per_article = [
#     build_vector_metadata(articles[i]["title"], graph_results[i].entities, graph_results[i].relations)
#     for i in range(len(articles))
# ]

In [ ]:
# Step 2: Embed each article and combine with metadata
#
# This is what you'd upsert to Pinecone / Qdrant / Weaviate:

# TODO: Embed each article
# vectors = []
# for i, article in enumerate(articles):
#     emb = client.embed.dense(article["text"], dimension=512)
#     vectors.append({
#         "id": f"article_{i}",
#         "embedding": emb.embeddings[0],
#         "metadata": metadata_per_article[i],
#     })
#
# # Show what one vector record looks like
# import json
# sample = {k: v for k, v in vectors[0].items() if k != "embedding"}
# sample["embedding"] = f"[{vectors[0]['embedding'][0]:.4f}, {vectors[0]['embedding'][1]:.4f}, ... ] (512-dim)"
# print(json.dumps(sample, indent=2))

In [ ]:
# Step 3: Filtered search demo
#
# Query: "Who leads Project Aurora?"
# Without metadata: cosine similarity alone — all 3 articles score similarly
# With metadata filter: has_people=True AND "Project Aurora" in entity_names
#   → narrows candidates BEFORE similarity, returning the right article

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def search_with_metadata(query_text, vectors, metadata_filter=None, top_k=3):
    """Simulate a vector DB search with optional metadata pre-filtering."""
    # Apply metadata filter first (this is what the vector DB does)
    candidates = vectors
    if metadata_filter:
        candidates = [v for v in vectors if metadata_filter(v["metadata"])]

    if not candidates:
        return []

    # Embed the query
    query_emb = client.embed.dense(query_text, dimension=512)
    query_vec = query_emb.embeddings[0]

    # Rank by cosine similarity
    scored = [
        (v["metadata"]["title"], cosine_similarity(query_vec, v["embedding"]))
        for v in candidates
    ]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

# TODO: Run search with and without filter, compare results
# query = "Who leads Project Aurora?"
#
# print("Without metadata filter (plain vector search):")
# for title, score in search_with_metadata(query, vectors):
#     print(f"  {score:.4f}  {title}")
#
# print("\nWith metadata filter (has_people + mentions Aurora):")
# for title, score in search_with_metadata(
#     query, vectors,
#     metadata_filter=lambda m: m["has_people"] and "Project Aurora" in str(m["entity_names"])
# ):
#     print(f"  {score:.4f}  {title}")

## 8. Graph-Augmented Embeddings

**The problem:** The raw text "The AI Research team, headed by Dr. Maria Santos, focuses on applied machine learning..." embeds as general text about AI research and machine learning. The resulting vector doesn't capture the *structural* knowledge that Maria Santos is a PERSON who heads a TEAM which built specific PROJECTS.

**The solution:** Before embedding, prepend a structured context block derived from the knowledge graph. The embedding model encodes both the raw semantic content *and* the relational structure into the same vector.

This matters most for queries that target relationships ("Who works on what?", "Which company signed the deal?") rather than topics ("Tell me about AI").

In [ ]:
# Build a graph context prefix from ontology results

def build_graph_context(entities, relations):
    """Generate a structured text prefix from knowledge graph output.
    
    This gets prepended to the raw text before embedding, so the vector
    captures both semantic content and relational structure.
    """
    lines = []
    if entities:
        entity_str = ", ".join(
            f"{e.text} ({e.label})" for e in entities[:10]
        )
        lines.append(f"Entities: {entity_str}")
    if relations:
        rel_str = "; ".join(
            f"{r.source_entity} -> {r.relation_type} -> {r.target_entity}"
            for r in relations[:10]
        )
        lines.append(f"Relations: {rel_str}")
    return "\n".join(lines)

# Example of what the augmented text looks like:
example_context = """Entities: Dr. Maria Santos (PERSON), Project Aurora (PROJECT), James Wright (PERSON), Raj Patel (PERSON), DataFlow Inc. (ORGANIZATION)
Relations: Dr. Maria Santos -> leads -> Project Aurora; James Wright -> greenlit -> Project Aurora; Raj Patel -> built -> inference pipeline"""

print("Graph context prefix (prepended before embedding):")
print(example_context)
print(f"\n+ original article text ({len(articles[0]['text'])} chars)")

In [ ]:
# TODO: Embed each article in two variants:
#   plain_embeddings[i]     = embed(article_text)
#   augmented_embeddings[i] = embed(graph_context + "\n\n" + article_text)
#
# plain_embeddings = []
# augmented_embeddings = []
#
# for i, article in enumerate(articles):
#     # Plain
#     plain = client.embed.dense(article["text"], dimension=512)
#     plain_embeddings.append(plain.embeddings[0])
#
#     # Augmented: prepend graph context
#     context = build_graph_context(
#         graph_results[i].entities,
#         graph_results[i].relations,
#     )
#     augmented_text = context + "\n\n" + article["text"]
#     augmented = client.embed.dense(augmented_text, dimension=512)
#     augmented_embeddings.append(augmented.embeddings[0])
#
# print(f"Embedded {len(articles)} articles in plain and augmented variants")

In [ ]:
# Compare retrieval quality: relational queries where structure matters
#
# These queries target specific relationships that plain text search
# struggles with — they need structural awareness to rank correctly.

relational_queries = [
    ("Who is responsible for the recommendation engine?", "Project Aurora - Q1 Launch"),
    ("Which projects does the AI Research team work on?", "Team Spotlight: AI Research"),
    ("What companies are involved in the $1.8M deal?", "Partnership Announcement: DataFlow Inc."),
]

# TODO: For each query, compute similarity against plain and augmented embeddings
# and compare ranking
#
# print(f"{'Query':<55} {'Expected':<20} {'Plain #1':<20} {'Augmented #1':<20}")
# print("-" * 115)
#
# for query_text, expected_title in relational_queries:
#     query_emb = client.embed.dense(query_text, dimension=512).embeddings[0]
#
#     # Rank against plain embeddings
#     plain_scores = [
#         (articles[i]["title"], cosine_similarity(query_emb, plain_embeddings[i]))
#         for i in range(len(articles))
#     ]
#     plain_scores.sort(key=lambda x: x[1], reverse=True)
#
#     # Rank against augmented embeddings
#     aug_scores = [
#         (articles[i]["title"], cosine_similarity(query_emb, augmented_embeddings[i]))
#         for i in range(len(articles))
#     ]
#     aug_scores.sort(key=lambda x: x[1], reverse=True)
#
#     short_query = query_text[:52] + "..." if len(query_text) > 55 else query_text
#     print(f"{short_query:<55} {expected_title[:18]:<20} {plain_scores[0][0][:18]:<20} {aug_scores[0][0][:18]:<20}")

### Interpretation

For general semantic queries ("tell me about machine learning"), both plain and augmented embeddings perform similarly — the base text already contains enough signal.

For **relational queries** ("who leads what", "which company signed the deal"), augmented embeddings surface the correct document more reliably because the structured entity and relation context gives the embedding model explicit signal about *who does what to whom*.

This is the practical ROI of the Relation Extraction service:
- **Metadata filtering** eliminates irrelevant candidates before cosine similarity runs (faster + more precise)
- **Graph-augmented embeddings** bake structural knowledge into the vector itself (better recall for relational queries)
- Both approaches use standard vector databases — no graph DB required

---
## Cleanup

In [ ]:
client.close()
print("Done.")